# Segment 3 — LangGraph: pause & resume

A LangGraph can **save its progress and pause** partway through, wait for a human, then continue.
Here it drafts a reply, pauses for approval, then sends.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## Build the graph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# State = a dict passed from step to step.
class State(TypedDict):
    request: str
    draft: str
    sent: str

def write_draft(state):
    draft = llm.invoke("Write a short reply to: " + state["request"]).content
    return {"draft": draft}

def send(state):
    return {"sent": "SENT: " + state["draft"]}

graph = StateGraph(State)
graph.add_node("write_draft", write_draft)
graph.add_node("send", send)
graph.add_edge(START, "write_draft")
graph.add_edge("write_draft", "send")
graph.add_edge("send", END)

# MemorySaver remembers progress; interrupt_before pauses before "send".
app = graph.compile(checkpointer=MemorySaver(), interrupt_before=["send"])

## See the graph as a diagram

In [ ]:
print(app.get_graph().draw_mermaid())

## Run — it stops right before sending

In [ ]:
config = {"configurable": {"thread_id": "ticket-1"}}
app.invoke({"request": "I was charged twice, please help."}, config)

print("Draft waiting for approval:")
print(app.get_state(config).values["draft"])

## A human approves → resume with `None`

In [ ]:
result = app.invoke(None, config)
print(result["sent"])